# Streamlit 画像分類アプリの Google Cloud Run へのデプロイ

このガイドでは、Streamlit を使用してシンプルな画像分類アプリケーションを構築するためのスキャフォールド（骨組み）と手順を説明します。


### Streamlit とは？
[Streamlit](https://streamlit.io/) は、特にデータサイエンスや機械学習プロジェクト向けのインタラクティブな Web アプリケーションを素早く作成するためのデファクトスタンダードとなっているオープンソースの Python ライブラリです。Streamlit の直感的な設計により、最小限のコードで強力なアプリを簡単に構築できます。

[Streamlit Gallery](https://streamlit.io/gallery) にある素晴らしいサンプルをチェックして、作成できるアプリケーションの幅広さを確認してみてください。データ ダッシュボードや可視化から、機械学習のデモやインタラクティブ ツールまで多彩です。

このノートブックはアクセラレータを必須としませんが、後述のトレーニング プロセスを高速化するためにオプションで GPU をアタッチできます。<br>
（GPU なしでトレーニングすると 15 分以上かかります。あるいは、アプリケーションのデプロイを試すだけであれば、トレーニングを途中で停止することも可能です。）

## セットアップ

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import base64
import json
import shutil
from datetime import datetime

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from google.cloud import aiplatform

## 画像分類モデルの構築

5 種類の花（5-Flower）データセットを例として使用し、花の画像分類モデルを作成します。

以下のコードについて詳しく知りたい場合は、[../../../notebooks/image_models/solutions/3_tf_hub_transfer_learning.ipynb](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/asl_core/notebooks/image_models/solutions/3_transfer_learning_fine_tuning.ipynb) を確認してください。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT + "-flowers"
FILE_DIR = f"gs://{BUCKET}/data"
REGION = "us-central1"

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3

BATCH_SIZE = 32

In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
    
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "Here are your current buckets:"
    gcloud storage ls
fi

create_tfrecords_at_scale.ipynb ノートブックをまだ実行していない場合は、下のセルのコメントを解除して gs://asl-public-data バケットからデータをコピーしてください。

In [ ]:
# !gcloud storage buckets create gs://{BUCKET}
# !gcloud storage cp gs://asl-public-data/data/flowers/tfrecords/* {FILE_DIR}

In [ ]:
!gcloud storage ls {FILE_DIR}

In [ ]:
TRAIN_PATTERN = FILE_DIR + "/train*"
EVAL_PATTERN = FILE_DIR + "/eval*"


def parse_example(example):
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }
    example = tf.io.parse_single_example(example, feature_description)
    example["image"] = tf.io.decode_jpeg(example["image"], channels=3)
    example["image"] = keras.ops.image.resize(
        example["image"], [IMG_HEIGHT, IMG_WIDTH]
    )
    return example["image"], example["label"]


train_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(TRAIN_PATTERN))
    .map(parse_example)
    .batch(BATCH_SIZE)
)
eval_ds = (
    tf.data.TFRecordDataset(tf.io.gfile.glob(EVAL_PATTERN))
    .map(parse_example)
    .batch(10)
)

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

transfer_model = keras.Sequential(
    [
        keras.Input(
            shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), name="mobilenet_input"
        ),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)

transfer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
transfer_model.fit(
    train_ds,
    epochs=5,
    validation_data=eval_ds,
)

In [ ]:
shutil.rmtree("export", ignore_errors=True)
os.mkdir("export")
# 通常のモデルのシリアル化には、.keras 拡張子を付けて .save() を使用できます
transfer_model.save("export/flowers_model.keras")

In [ ]:
!ls export/flowers_model.keras

---

## Streamlit アプリケーションの構築

画像分類モデルの準備ができたので、インタラクティブでユーザーフレンドリーな Streamlit アプリを作成しましょう。

### ライブラリのインポート
まず、Streamlit 自体を含む必要な Python ライブラリと、画像処理およびモデルの読み込みに必要なその他のライブラリをインポートします。

In [ ]:
%%writefile app.py
"""Streamlit 画像分類アプリ"""

import keras
import numpy as np
import streamlit as st
import tensorflow as tf
from keras.models import load_model

### テキスト要素の追加
Streamlit は最終的に Web ページをレンダリングしますが、Python モジュールを使用して Web ページの要素を定義・構成するだけで済みます。

まず、アプリのいくつかのメタデータを構成します。

- [`st.set_page_config`](https://docs.streamlit.io/develop/api-reference/configuration/st.set_page_config) を使用すると、ページ タイトルやファビコン（ブラウザのタブに表示される小さなアイコン）などをカスタマイズできます。

Streamlit にはテキスト コンテンツを表示するためのさまざまな方法が用意されています。
- [`st.title`](https://docs.streamlit.io/develop/api-reference/text/st.title) は、ページにメインの見出しを追加するために使用します。
- より小さな見出しには [`st.header`](https://docs.streamlit.io/develop/api-reference/text/st.header) や [`st.subheader`](https://docs.streamlit.io/develop/api-reference/text/st.subheader) を使用することもできます。
- [`st.text`](https://docs.streamlit.io/develop/api-reference/text/st.text) はプレーンテキストを表示するためのものです。
- [`st.markdown`](https://docs.streamlit.io/develop/api-reference/text/st.markdown) を使用すると、Markdown 構文を使用して書式設定されたテキストを追加できます。
その他のオプションについては、Streamlit ドキュメントの [その他のテキスト要素](https://docs.streamlit.io/develop/api-reference/text) を確認してください。

Streamlit には、「万能ツール」とも言える [`st.write`](https://docs.streamlit.io/develop/api-reference/write-magic/st.write) コマンドも用意されています。テキスト、DataFrame（データ テーブル）、Matplotlib プロット、さらには Keras 機械学習モデルなど、多くの種類のコンテンツを処理できます。

また、ここでは後で使用するいくつかのグローバル変数も定義します。

In [ ]:
%%writefile -a app.py

st.set_page_config(page_title="5-Flower Classifier", page_icon="🌷")

st.title("5-Flower Classifier")

st.markdown(
    "Welcome to this simple web application that classifies 5 flowers"
    + "(daisy, dandelion, roses, sunflowers, tulips)."
)

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_CHANNELS = 3
CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

### キャッシュを使用したモデル読み込みの定義

`keras.models.load_model` を使用して画像分類モデルを読み込む関数を定義します。

予測を行うたびにモデルを再読み込みするのを避けるため、[`@st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource) デコレータを使用します。このデコレータは関数の出力をキャッシュするため、次回以降の実行でモデルへのアクセスが大幅に高速化されます。

**注: [`@st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource) は、データベース接続や複雑な機械学習モデルなど、簡単にシリアル化（単純なデータ形式に変換）できないグローバル オブジェクトに最適です。より単純でシリアル化可能なオブジェクト（pandas DataFrame など）の場合は、代わりに [`tf.cache_data`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_data) の使用を検討してください。**

キャッシュの詳細と Streamlit アプリの最適化方法については、[こちらのドキュメント](https://docs.streamlit.io/develop/concepts/architecture/caching) を確認してください。

In [ ]:
%%writefile -a app.py

@st.cache_resource(show_spinner=False)
def load_and_cache_model():
    model_path = "flowers_model.keras"
    model = load_model(model_path)
    return model


### ユーティリティ関数の定義

いくつかのユーティリティ関数も定義します。

モデル用の画像を準備し、その予測を解釈するために、画像のデコードやリスケール、リサイズ、後処理など、モデル サービング用のヘルパー関数をいくつか定義します。

In [ ]:
%%writefile -a app.py

def read_image(img_bytes):
    img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
    return img


def predict(model, image):
    image = keras.ops.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = np.expand_dims(image, axis=0)
    predictions = model.predict(image)
    pred_index = np.argmax(predictions[0])
    return predictions[0][pred_index], CLASSES[pred_index]

### アプリケーション ロジックの定義

それでは、メインのアプリケーション ロジックを定義しましょう。

以下の機能を備えたアプリケーションを設計します。
1. **画像のアップロード**: [`st.file_uploader`](https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader) を使用して、ユーザーが画像ファイルを選択してアップロードできるウィジェットを作成します。ユーザーを案内するために許可されるファイル形式（PNG や JPG など）を指定できます。
2. **アップロードされた画像の表示**: 画像がアップロードされたら、[`st.image`](https://docs.streamlit.io/develop/api-reference/media/st.image) を使用して表示します。あるいは、汎用的な [`tf.write`](https://docs.streamlit.io/develop/api-reference/write-magic/st.write) 関数でも画像の表示を処理できます。
3. **画像分類の開始**: [`st.button`](https://docs.streamlit.io/develop/api-reference/widgets/st.button) を使用して「Classify」というラベルのボタンを作成します。このボタンは押されると `True` を返します。（ステートフル ボタンなどのより高度なボタンのユースケースについては、Streamlit ドキュメントを参照してください: https://docs.streamlit.io/develop/concepts/design/buttons）。
4. **分類モデルの呼び出し**: 先ほど定義した関数を使用してモデルを読み込み、予測を行います。これには少し時間がかかる場合があるため、[`with st.spinner`](https://docs.streamlit.io/develop/api-reference/status/st.spinner) ブロックでラップして、処理が進行中であることをユーザーに視覚的にフィードバックします。
5. **結果の表示**: モデルが予測を生成したら、それをユーザーに表示します。[`st.success`](https://docs.streamlit.io/develop/api-reference/status/st.success) を使用して分類が正常に完了したことを明確に示し、予測された花の種類を表示します。

In [ ]:
%%writefile -a app.py

def main():
    file_uploaded = st.file_uploader("Choose File", type=["png", "jpg", "jpeg"])
    if file_uploaded is not None:
        image = read_image(file_uploaded.read())
        st.image(image.numpy(), caption="Uploaded Image")
        class_btn = st.button("Classify")
        if class_btn:
            with st.spinner("Model predicting...."):
                loaded_model = load_and_cache_model()
                prob, prediction = predict(loaded_model, image)
                st.success(f"Prediction: {prediction} - {prob:.2%}")


if __name__ == "__main__":
    main()

## Cloud Run へのアプリのデプロイ
Streamlit アプリケーションの準備が整いました。コンテナ化されたアプリケーションをシームレスに実行できるように設計されたサーバーレス プラットフォームである Google Cloud Run にデプロイしましょう。

### Dockerfile と依存関係の定義
アプリを Cloud Run 向けにコンテナ化するために、`requirements.txt` と `Dockerfile` を定義します。

In [ ]:
%%writefile requirements.txt
numpy==1.26.4
streamlit==1.56.0
tensorflow[and-cuda]==2.18.1
keras==3.12.0
tf_keras==2.18.0
keras_hub==0.21.1

In [ ]:
%%writefile Dockerfile
FROM python:3.12

WORKDIR /app

COPY requirements.txt /app
RUN pip install -r requirements.txt

COPY export /app
COPY app.py /app

EXPOSE 8080

CMD streamlit run --server.port 8080 --server.enableCORS false app.py

**注: `COPY` コマンドを複数の行に分割し、それぞれ異なるファイルをコピーしています。これは必須ではありませんが、Docker のキャッシュ メカニズムにとって極めて重要な最適化です。<br> app.py のみを変更した場合、次回イメージをビルドする際に Docker は依存関係のインストールやその他のファイルのキャッシュされたレイヤを再利用するため、ビルド プロセスが大幅に高速化されます。**

### コンテナのビルドと Artifact Registry への push
`Dockerfile` が用意できたので、Streamlit アプリの Docker イメージをビルドし、Google Cloud の Artifact Registry に push できます。
Artifact Registry は、コンテナ イメージを保存するための安全でスケーラブルな方法を提供します。

まず、コンテナ イメージを格納するために Artifact Registry に新しいリポジトリを作成します。

In [ ]:
STREAMLIT_ARTIFACT_REG_REPO = "flower-classification-app"
os.environ["STREAMLIT_ARTIFACT_REG_REPO"] = STREAMLIT_ARTIFACT_REG_REPO

In [ ]:
%%bash
if ! gcloud artifacts repositories describe $STREAMLIT_ARTIFACT_REG_REPO \
       --location=$REGION > /dev/null 2>&1; then
    gcloud artifacts repositories create $STREAMLIT_ARTIFACT_REG_REPO \
        --project=$PROJECT --location=$REGION --repository-format=docker
fi

### Cloud Build 用 cloudbuild.yaml の定義
Google Cloud Build を使用して、Docker イメージのビルドと Artifact Registry への push のプロセスを自動化します。

Cloud Build は、cloudbuild.yaml という構成ファイルでビルドステップを定義できるサーバーレス CI/CD プラットフォームです。

In [ ]:
CONTAINER_PATH = (
    f"us-central1-docker.pkg.dev/{PROJECT}/{STREAMLIT_ARTIFACT_REG_REPO}/app"
)
os.environ["CONTAINER_PATH"] = CONTAINER_PATH

ビルドを高速化するためにキャッシュを組み込んだ cloudbuild.yaml ファイルを以下に示します。

1. **既存イメージの pull**: 最初のステップでは、Artifact Registry から最新バージョンの Docker イメージの pull を試みます。ここではエントリポイントとして `bash -c` コマンドを使用しているため、最初の実行でこのステップが失敗した場合でもジョブは無視して続行できます。
2. **キャッシュを使用したビルド**: 2 番目のステップで新しいイメージをビルドします。`--cache-from` フラグは、pull したイメージのレイヤをキャッシュとして使用するよう Docker に指示し、それらのレイヤに変更がない場合にビルドを高速化します。

In [ ]:
%%writefile cloudbuild.yaml
steps:
- name: 'gcr.io/cloud-builders/docker'
  entrypoint: 'bash'
  args: ['-c', 'docker pull ${_CONTAINER_PATH}:latest || exit 0']
- name: 'gcr.io/cloud-builders/docker'
  args: [
            'build',
            '-t', '${_CONTAINER_PATH}:latest',
            '--cache-from', '${_CONTAINER_PATH}:latest',
            '.'
        ]
images: ['${_CONTAINER_PATH}:latest']

### コンテナ イメージのビルド

cloudbuild.yaml ファイルが定義されたので、Docker イメージをビルドするよう Cloud Build に指示できます。

In [ ]:
!gcloud builds submit --config cloudbuild.yaml --region $REGION . --substitutions _CONTAINER_PATH={CONTAINER_PATH}

### Cloud Run へのデプロイ
Artifact Registry に保存されたコンテナ イメージの準備が整ったので、Streamlit アプリを Cloud Run にデプロイする準備がすべて整いました。

このコマンドを上記で定義した `cloudbuild.yaml` に組み込むことも検討できます。

In [ ]:
APP_NAME = "flower-classification"
os.environ["APP_NAME"] = APP_NAME

In [ ]:
%%bash
echo 'Deploying the application to Cloud Run...'
gcloud run deploy $APP_NAME \
  --image $CONTAINER_PATH:latest --min-instances 1 --max-instances 1 --cpu 1 \
  --memory 4Gi --region us-central1 > /dev/null 2>&1 && \
echo 'Deployment Done.'

### Cloud Shell 経由での Cloud Run アプリへの接続


Cloud Run サービスへのアクセス構成には、多くの柔軟性があります。必要に応じて、[一般公開する](https://cloud.google.com/run/docs/authenticating/public) ことも可能です。

ただし、この例では、プロキシを使用して Cloud Shell から Cloud Run アプリに安全に接続する方法を見ていきましょう。

Cloud Shell からアプリを開くには、次の手順に従います。
1. 次のセルを実行し、出力された `gcloud run services proxy ...` コマンドをコピーします。
2. Cloud Shell を開き、コマンドを貼り付けて実行します。
3. Cloud Shell のツールバーにある「ウェブでプレビュー（Web Preview）」ボタンをクリックします。
4. 「ポート 8080 でプレビュー（Preview on port 8080）」を選択します。
5. 新しいブラウザのタブまたはウィンドウが開き、Streamlit アプリが表示されます。

In [ ]:
print(
    f"gcloud run services proxy {APP_NAME} --project {PROJECT} --region {REGION}"
)

## 予測
これでモデルをテストする準備が整いました。

モデルが認識する花の種類（ヒナギク、タンポポ、バラ、ヒマワリ、チューリップ）の鮮明な画像を検索し、画像をアップロードして「Classify」をクリックします。<br>
アプリに予測された花の種類が表示されます。

最初のアクセスと予測には少し時間がかかる場合がありますが、キャッシュのおかげで 2 回目以降はより高速になります。

<img width="1004" alt="image" src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/4b6fd594-69fd-491e-9ca2-c199d3cb66a7">

Copyright 2024 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.